# RHNA & Housing Production

RHNA targets and housing production (permits/completions) for the 18
incorporated jurisdictions in San Diego County plus the County itself,
pulled from HCD, DOF, City of San Diego, and Census sources.

Target year: 2025 for RHNA/APR/DOF/permits. ACS uses the 2020-2024
5-year vintage (its most recent release).


## Setup

In [ ]:
%pip install pandas numpy requests openpyxl

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
        my_folder = candidate / "jake's work"
        if (my_folder / "data").exists() and (my_folder / "notebooks").exists():
            return my_folder
    raise FileNotFoundError("Could not locate workstream root.")


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Workstream root:", ROOT)


In [ ]:
TARGET_YEAR = 2025
ACS_VINTAGE_LABEL = "2020-2024"
ACS_DATA_YEAR = 2024

SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]
COUNTY_JURISDICTION_NAME = "San Diego County"


In [ ]:
def normalize_jurisdiction(name: object) -> str:
    s = str(name).strip().lower()
    if s == "national city":
        return "national city"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation"},
    )
    if not response.ok:
        raise RuntimeError(f"Request failed ({response.status_code}): {response.url}")
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str):
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        raise ValueError(f"No CSV resource matching '{name_contains}' found.")
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


## APR (permits, entitlements, completions)

In [ ]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)
package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")

raw_path = RAW_DIR / "apr_table_a2_raw.csv"
if not raw_path.exists():
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


In [ ]:
print(apr_raw.columns.tolist())


In [ ]:
JURISDICTION_COL = "Jurisdiction Name"  # confirm against columns above
YEAR_COL = "Reporting Year"              # confirm against columns above

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)
apr_raw[YEAR_COL] = pd.to_numeric(apr_raw[YEAR_COL], errors="coerce")

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_target_year))
missing = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


In [ ]:
BP_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("BP_") and "INCOME" in c.upper()]
CO_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("CO_") and "INCOME" in c.upper()]

sd_apr_target_year["bp_units_total"] = sd_apr_target_year[BP_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["co_units_total"] = sd_apr_target_year[CO_INCOME_COLS].sum(axis=1, numeric_only=True)

above_mod_bp = [c for c in BP_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_co = [c for c in CO_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_target_year["bp_affordable_total"] = (
    sd_apr_target_year["bp_units_total"] - sd_apr_target_year[above_mod_bp].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["co_affordable_total"] = (
    sd_apr_target_year["co_units_total"] - sd_apr_target_year[above_mod_co].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["bp_affordable_share"] = sd_apr_target_year["bp_affordable_total"] / sd_apr_target_year["bp_units_total"]
sd_apr_target_year["co_affordable_share"] = sd_apr_target_year["co_affordable_total"] / sd_apr_target_year["co_units_total"]

production_by_year = sd_apr_target_year[[
    "jur_clean", YEAR_COL, "bp_units_total", "co_units_total",
    "bp_affordable_total", "co_affordable_total",
    "bp_affordable_share", "co_affordable_share",
]].rename(columns={YEAR_COL: "year"})
production_by_year.head()


## RHNA 6th Cycle targets

In [ ]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"
rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


In [ ]:
print(rhna_raw.columns.tolist())

In [ ]:
RHNA_JUR_COL = "Jurisdiction"  # confirm against columns above

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("SD rows:", len(sd_rhna6))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_rhna6.head()


## City of San Diego permits

In [ ]:
PERMITS_PACKAGE_URL = "https://data.sandiego.gov/api/3/action/package_show?id=development-permits-set2"
permits_package_json = get_json(PERMITS_PACKAGE_URL)
active_url, active_name = find_resource_download_url(permits_package_json, "Active approvals")
closed_url, closed_name = find_resource_download_url(permits_package_json, "Closed approvals")

permits_frames = []
for label, url in [("active", active_url), ("closed", closed_url)]:
    raw_path = RAW_DIR.parent / "sandiego_permits" / f"{label}_approvals_raw.csv"
    raw_path.parent.mkdir(parents=True, exist_ok=True)
    if not raw_path.exists():
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        raw_path.write_bytes(resp.content)
    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
print(sd_permits_raw.shape)
sd_permits_raw.head()


In [ ]:
print(sd_permits_raw.columns.tolist())

## CA DOF population & housing estimates (E-5)

In [ ]:
DOF_RAW_PATH = RAW_DIR.parent / "dof" / "e5_population_housing.xlsx"
DOF_SHEET_NAME = f"E5CityCounty{TARGET_YEAR}"

dof_raw = pd.read_excel(DOF_RAW_PATH, sheet_name=DOF_SHEET_NAME, header=3)
dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
dof_raw["name"] = dof_raw["name"].astype(str).str.strip()

dof_raw["is_county_header"] = dof_raw["Total"].isna() & dof_raw["name"].str.contains("County", na=False)
dof_raw["county"] = dof_raw["name"].where(dof_raw["is_county_header"]).ffill()

sd_dof = dof_raw[
    (dof_raw["county"] == "San Diego County")
    & (~dof_raw["is_county_header"])
    & (dof_raw["Total"].notna())
    & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
].copy()

sd_dof["jur_clean"] = sd_dof["name"].map(
    lambda n: "county san diego" if n.strip() == "Unincorporated" else normalize_jurisdiction(n)
)
sd_dof["year"] = TARGET_YEAR

print("SD rows:", len(sd_dof))
missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_dof[["name", "Total", "Household", "Group Quarters"]]


## Census ACS (2020-2024 5-year estimates)

In [ ]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_VINTAGE_LABEL}):", len(sd_acs))
sd_acs.head()


## Summary

In [ ]:
loaded = {
    "APR (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "City of SD permits": sd_permits_raw if "sd_permits_raw" in dir() else pd.DataFrame(),
    "DOF (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}
for name, df in loaded.items():
    if df.empty:
        print(f"{name:30s} not loaded")
    else:
        n_missing = (df.isna().sum() > 0).sum()
        print(f"{name:30s} {df.shape[0]} rows, {df.shape[1]} cols, {n_missing} cols with missing values")


## Metric dictionary

In [ ]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "bp_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns",
        "definition": "Total housing units with a building permit issued, all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "co_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns",
        "definition": "Total housing units with a certificate of occupancy (completed), all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
])
metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


## Export

In [ ]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
production_by_year.to_csv(output_path, index=False)
print("Saved:", output_path)


## Validation against dashboard prototype

In [ ]:
BASELINE_PATH = Path(
    "../../housing-dashboard-prototype/data/processed/sd_apr_a2_city_year_supply.csv"
)

if BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    comparison = production_by_year.merge(
        baseline, on="jur_clean", suffixes=("_fresh", "_baseline"), how="inner",
    )
    failed_checks = comparison[
        comparison["bp_units_total_fresh"] != comparison["bp_units_total_baseline"]
    ]
    print(f"Compared {len(comparison)} rows; {len(failed_checks)} mismatches.")
    display(failed_checks[["jur_clean", "bp_units_total_fresh", "bp_units_total_baseline"]])
    comparison.to_csv(PROCESSED_DIR / "rhna_housing_production_validation_report.csv", index=False)
else:
    print(f"Baseline not found at {BASELINE_PATH}.")


## Notes

- RHNA targets come from Table B (via the separate RHNA progress dataset), not Table A2.
- Table A2 contains entitlements, permits, and certificates of occupancy as
  separate date fields on the same project row -- a project can appear as
  entitled in one year and permitted in a different year.
- ACS is pinned to 2024 (2020-2024 vintage) while other sources target 2025;
  any table joining ACS onto 2025 production data mixes two data years.
- `normalize_jurisdiction()` special-cases "National City" -- a blind
  trailing-"city" strip would otherwise collapse it to "national".
